# soulclip on Colab — free AI video from a scene script

Turns a scene-by-scene script into a stitched film using **Wan 2.1 T2V 1.3B**
on Colab's free T4 GPU. No API key, no payment.

**Before you start:** `Runtime > Change runtime type > T4 GPU`. Without a GPU
this will not run.

### How long it really takes

Generation dominates everything else. On a free T4, one 5-second 480p clip
takes roughly **6-10 min at 20 steps**, or **3-5 min at 10 steps**.

| Clips | Film | 10 steps | 20 steps |
|---|---|---|---|
| 6 | 30 s | 20-30 min | 35-60 min |
| 12 | 1 min | 35-60 min | 1.2-2 hrs |
| 24 | 2 min | 1.2-2 hrs | 2.5-4 hrs |
| **60** | **5 min** | **3-5 hrs** | **6-10 hrs** |

Add ~5 min once per session for the model download, and ~3 min for the final
stitch (measured, not estimated).

Colab disconnects free sessions after roughly 90 minutes, so a 5-minute film
means **2-3 sessions at 10 steps, or 4-7 at 20 steps**. That is fine — every
clip is saved as it finishes, so re-running continues from where it stopped.
Step 3 mounts Google Drive so progress survives a disconnect.

**Start with 6 clips.** You will see real generated video in under half an
hour and can judge the quality before committing to a long run.


## 1. Check the GPU


In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), (
    'No GPU. Runtime > Change runtime type > T4 GPU, then rerun.')
p = torch.cuda.get_device_properties(0)
print(f'{p.name}, {p.total_memory/1e9:.1f} GB VRAM')


## 2. Install

Takes 2-3 minutes.


In [ ]:
!pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg
!git clone -q https://github.com/Naserkhan07/soul_exter.git 2>/dev/null || true
%cd /content/soul_exter
!git checkout -q arena/019f98a2-soul-exter && git pull -q
print('ready')


## 3. Keep your work across disconnects (recommended)

Saves clips to Drive so a dropped session costs you nothing. Skip this
cell if you would rather not connect Drive — but then a disconnect loses
everything generated so far.


In [ ]:
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKDIR = '/content/drive/MyDrive/soulclip/work'
    OUTPUT  = '/content/drive/MyDrive/soulclip/film.mp4'
else:
    WORKDIR = '/content/work'
    OUTPUT  = '/content/film.mp4'

print('clips ->', WORKDIR)
print('film  ->', OUTPUT)


## 4. Your script

Label scenes `Scene 1:`, `Scene 2:` ... or separate them with blank lines.

**Prompt tips for Wan:** describe the *camera* and the *motion*, not just
the subject — 'slow dolly in', 'waves crash', 'hair moves in the wind'.
Repeat character details in every scene; the model has no memory between
clips.


In [ ]:
script = '''
Scene 1: A lighthouse on a black rock headland at dusk, its beam sweeping
slowly across heavy grey water. Rain streaks sideways. Slow dolly in.

Scene 2: Inside the lantern room, brass fittings glowing warm. An old
keeper in a wool coat winds a mechanism by hand. Firelight flickers.

Scene 3: Waves crash white over a dark reef, spray flung high into the
storm. Handheld camera, violent motion.

Scene 4: A small fishing boat pinned against the rocks, mast broken, a
single lantern swinging wildly on the deck.

Scene 5: The keeper hauls a heavy lever with both hands, straining. The
great beam swings and holds steady.

Scene 6: Dawn over a calm flat sea, pale gold light. Two figures wrapped
in blankets sit on stone steps, steam rising from tin mugs.
'''

with open('/content/script.txt', 'w') as f:
    f.write(script)

!python -m soulclip.cli scenes /content/script.txt --clip-seconds 5


## 5. Settings

`CLIPS` is the main dial. Each clip is ~5 s, so 6 clips = 30 s of film.

Leave `STEPS` at 20 for good quality, or drop to 10 to roughly halve the
time at some cost in sharpness.


In [ ]:
CLIPS = 6      # 6 = 30s film. 60 = 5 minutes (many hours, several sessions)
STEPS = 10     # 10 is ~2x faster and a bit softer; 20 for best quality
WIDTH, HEIGHT = 832, 480

STYLE = 'cinematic anime, detailed background art, dramatic lighting, film grain'

mins = CLIPS * 5
lo, hi = (3, 5) if STEPS <= 10 else (6, 10)
print(f'{CLIPS} clips x ~5s = ~{mins//60}m {mins%60:02d}s of film')
print(f'estimated {CLIPS*lo/60:.1f}-{CLIPS*hi/60:.1f} hours of generation on a T4')
print(f'(plus ~5 min model download and ~3 min stitching)')


## 6. Generate

The first run downloads ~6 GB of weights (a few minutes, once per session).

**If it disconnects, just run this cell again** — finished clips are reused
and only the missing ones are generated.


In [ ]:
!python -m soulclip.cli render /content/script.txt \
    --provider wan \
    --clip-seconds 5 \
    --max-scenes $CLIPS \
    --target $((CLIPS*5)) \
    --wan-steps $STEPS \
    --width $WIDTH --height $HEIGHT \
    --style "$STYLE" \
    --crossfade 0.4 \
    --workdir "$WORKDIR" \
    -o "$OUTPUT"


## 7. Watch it


In [ ]:
from IPython.display import HTML
from base64 import b64encode

data = b64encode(open(OUTPUT, 'rb').read()).decode()
HTML(f'<video width=640 controls src="data:video/mp4;base64,{data}"></video>')


## 8. Download


In [ ]:
from google.colab import files
files.download(OUTPUT)


---
## Going to a full 5 minutes

Set `CLIPS = 60` in step 5 and re-run step 6. At 10 steps expect **3-5 hours**
of generation spread over 2-3 sessions; at 20 steps, 6-10 hours over 4-7.

With Drive mounted the loop is:

1. Run step 6 until Colab disconnects
2. Reconnect, re-run steps 1-3, then step 6 again
3. Repeat — each pass adds more clips and reuses everything already done

The final stitch runs automatically once all 60 clips exist.

**Note on length:** crossfades overlap the clips, so 60 x 5 s with 0.4 s
fades gives about 4m40s, not 5m00s. Use `CLIPS = 64` for a true five
minutes, or drop `--crossfade` for hard cuts.

### If you hit out-of-memory

Add `--wan-frames 33` (≈2 s clips) or lower the resolution to `640x384`.

### Honest expectations

Wan 1.3B is the smallest real video model. Motion is genuine but the output
is 480p and less polished than paid Kling or Veo. Characters will not stay
consistent between shots — that is the model, not the pipeline. Repeating
the same character description in every scene helps a little.
